# Generation

## install
pip install rank-bm25

In [9]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)

texts = [c["text"] for c in chunks]

print("Чанков:", len(chunks), "| Запросов:", len(queries))
print("Смысловой запрос:", queries[1])
print("Запрос с точным термином:", queries[7])              # Индексы чанков как int

Чанков: 176 | Запросов: 66
Смысловой запрос: {'query': 'какая версия webOS или Tizen нужна для приложения на телевизоре', 'kind': 'exact', 'relevant_docs': [146], 'gold': 'LG с webOS 4.0 и новее, Samsung на Tizen с моделей 2018 года'}
Запрос с точным термином: {'query': 'сколько дней длится пробный период', 'kind': 'exact', 'relevant_docs': [155], 'gold': '14 дней бесплатного доступа к тарифу «Стандарт», один раз на аккаунт'}


In [20]:
'''
Retrieve - dict
'''
from sentence_transformers import SentenceTransformer
import chromadb

THRESHOLD = 0.83

client = chromadb.PersistentClient(path="chroma_kb") 
col = client.get_collection(name="knowledge_base")    


embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=5, threshold=THRESHOLD):
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11681.17it/s]


In [18]:
ret = retrieve_dict('хочу убрать из ленты фильмы, которые уже смотрел')
print(ret)

хочу убрать из ленты фильмы, которые уже смотрел
[{'id': 40, 'doc_id': 46, 'similarity': 0.8448, 'text': 'Краткая памятка по оценкам: ставьте «Нравится» фильмам, которые хочется советовать друзьям, и «Не нравится» тем, что не досмотрели. Так подборки быстрее поймут ваш вкус. Оценки можно менять сколько угодно раз — рекомендации перестроятся автоматически. Оценивать стоит и старые просмотры из истории: это тоже сигнал для подборок.'}]


In [25]:
'''
Generate
'''
from openai import OpenAI

client_openai = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=200):
    """Один вызов локальной языковой модели. temperature=0 — для
    воспроизводимости, режим рассуждения Qwen3.5 отключён"""
    resp = client_openai.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [40]:
SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def build_messages(query, chunk_dicts):
    """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
    # print('>>> chunk_dicts: ', chunk_dicts)
    context = "\n".join(f"[{n}] {i['text']}" for n, i in enumerate(chunk_dicts, 1))
    user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

def rag_answer(query, search_fn, k=3):
    """Полный RAG: retrieval -> промпт -> генерация."""
    ret = search_fn(query, k=3)
    msg = build_messages(query, ret)
    answer = generate(msg)
    return answer, [i['doc_id'] for i in ret], [i['text'] for i in ret]

In [ ]:
for q in [x for x in queries if x["kind"] == "semantic"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict, k=3)
  
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")


В: вечером фильмы постоянно тормозят и крутится кружок загрузки, что делать
   чанки: [177, 101, 112]  (['Скачивание фильмов и сериалов срыв', '# Фильм тормозит или прерывается Е', 'Если изображение идёт без звука ил'])
   О: Вечером сеть часто перегружена, поэтому рекомендуется вручную снизить качество видео в настройках плеера. Также проверьте реальную скорость интернета на устройстве и закройте другие приложения, которые используют сеть.

В: включаю кино — на экране всё идёт, а из колонок тишина
   чанки: [112, 91, 132]  (['Если изображение идёт без звука ил', 'Киновечер — это не «включить что-н', 'Чтобы фильмы выглядели на телевизо'])
   О: Проверьте настройки аудиодорожек в плеере и убедитесь, что выбранная дорожка не выключена и громкость не убавлена. Также проверьте громкость телевизора и подключенных колонок, а при необходимости перезапустите приложение.

В: телевизор показывает чёрный экран, но звук идёт
   чанки: [125, 110, 129]  (['Если при воспроизведении звук идет', 'Если

In [43]:
import re
from collections import defaultdict
from rank_bm25 import BM25Okapi

def tokenize(text):
    """Разбивает текст на токены-слова в нижнем регистре — вход для BM25."""
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(t) for t in texts])


def sparse_search(query, k=3):
    """Разреженный поиск BM25: top-k по совпадению слов с весом по редкости."""
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][:k]
    return [int(i) for i in order]


def rrf_fuse(ranked_lists, rrf_k=60):
    """Reciprocal Rank Fusion: сливаем выдачи по рангам."""
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[doc] += 1.0 / (rrf_k + rank)
    return [doc for doc, _ in sorted(scores.items(), key=lambda x: -x[1])]


def hybrid_search(query, k=3):
    """Гибридный поиск: сливаем плотную и разреженную выдачи через RRF."""
    fused = rrf_fuse([dense_search(query, 15), sparse_search(query, 15)])
    return fused[:k]


def is_hit(chunk_ids, relevant_ids):
    """Успех, если релевантный чанк дошёл до контекста (retrieval-hit) —
    метрика из задания 4."""
    return any(i in relevant_ids for i in chunk_ids)


methods = {"dense": retrieve_dict, "BM25": sparse_search}
# methods = {"dense": retrieve_dict, "BM25": sparse_search, "hybrid": hybrid_search}
score = {m: 0 for m in methods}
print(f"{'тип':10}{'запрос':46}" + "".join(f"{m:>8}" for m in methods))
for q in queries[:3]:
    row = f"{q['kind']:10}{q['query'][:44]:46}"
    for m, fn in methods.items():
        answer, ids, *_ = rag_answer(q["query"], fn, k=3)
        ok = is_hit(ids, q["relevant_docs"])
        score[m] += ok
        row += f"{('верно' if ok else '—'):>8}"
    print(row)

n = len(queries)
print("\nВерных ответов из", n, ":", {m: score[m] for m in methods})

тип       запрос                                           dense    BM25


TypeError: 'int' object is not subscriptable